# 13a · Vendor plus the three code columns

12b showed the vendor marker was worth **+0.0247** and moved the
`unseen in training` bucket from 0.6262 to 0.6745. This adds the three
columns the new pull brings back, by the same mechanism.

| column | what it is |
|---|---|
| `ICSP Prodcat` | the ERP product category |
| `UNSPSC` | the standard commodity code, 8 digits |
| `Prodline` | product line, `cono` + line |

**UNSPSC is emitted at four granularities**, not just whole. The code
nests segment / family / class / commodity, so a single eight-digit
token is all-or-nothing: useless the moment the model meets a
commodity it has not seen. Emitting `UNSPSC2_46`, `UNSPSC4_4618`,
`UNSPSC6_461815` and `UNSPSC8_46181504` lets it fall back to the
segment — which is exactly the bucket carrying 74% of the remaining
error.

**The cleaning rules are unchanged**, deliberately. This measures the
new columns alone, so the next experiment can measure the regex
changes against a stable baseline rather than a moving one.

Writes to `outputs_codes/` and `cleaned_data_codes.txt`. 12a's files
are untouched, so 12b's **0.8036** stays reproducible as the
comparator.

In [0]:
import os, sys, pathlib

# Isolation, as in 11a. This arm writes its own cleaned file, its own
# hierarchy lookup and its own outputs directory, so it cannot disturb
# the shared files, 11a's vendor files, or any existing bundle.
#
# data_dir() is deliberately NOT redirected -- the cleaning reference
# lists live there -- so every file written into it gets a distinct
# name, asserted before each write.
ROOT = pathlib.Path.cwd()
if not (ROOT / "nn_classifier").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

ARM_OUTPUT_DIR = str(ROOT / "outputs_codes")
os.environ["NN_CLASSIFIER_OUTPUT_DIR"] = ARM_OUTPUT_DIR
pathlib.Path(ARM_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

CLEAN_FILE = "cleaned_data_codes.txt"
LOOKUP_FILE = "hierarchy_lookup_codes.csv"
for name in (CLEAN_FILE, LOOKUP_FILE):
    assert name not in ("cleaned_data.txt", "hierarchy_lookup.csv",
                        "cleaned_data_vendor.txt", "hierarchy_lookup_vendor.csv",
                        "cleaned_data_gvendor.txt",
                        "hierarchy_lookup_gvendor.csv")

import numpy as np
import pandas as pd

import nn_classifier as nnc
from nn_classifier import paths

assert str(paths.output_dir()) == ARM_OUTPUT_DIR
print(paths.describe())


def show(frame, empty_msg="(nothing to show)"):
    """display() that survives an empty frame.

    Databricks converts a pandas frame to Spark to render it, and an
    empty one raises CANNOT_INFER_EMPTY_SCHEMA rather than drawing an
    empty table. Every display below is of a *filtered* frame, so any
    of them can be empty -- and an empty result is a finding, not a
    crash.
    """
    if len(frame) == 0:
        print(empty_msg)
    else:
        display(frame)

[paths] scratch: /local_disk0/tmp (cluster SSD): PermissionError -- using /tmp/nn_classifier
environment: Databricks
  data dir:   /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/data
  output dir: /Workspace/Users/patrick.white@vallen.com/Category Classification/NN Category Classifier/outputs_codes
  scratch:    /tmp/nn_classifier  (118 GB free)
  device:     cpu


In [0]:
from nn_classifier import cleaning, data
from nn_classifier.config import (DataConfig, CleaningConfig, FilterConfig,
                                  RunConfig, save_config)

RAW_FILE = "raw_data.txt"
data_cfg = DataConfig()

# Fail here rather than four cells later with an empty frame.
#
# Databricks keeps the interpreter alive between runs, so `import
# nn_classifier` does NOT re-read a file that changed on disk. If the
# repo synced after this session first imported the package, the old
# DataConfig class is still in memory and vendor_col is still the
# product number -- which silently rebuilds 11b's broken dataset.
#
# The symptom is 'mean rows per vendor: 1' two cells down, followed by
# an empty frame and CANNOT_INFER_EMPTY_SCHEMA. Easy to misread as a
# data problem.
import inspect

import nn_classifier.config as _cfgmod
from nn_classifier import cleaning as _cl
from nn_classifier.cleaning import descriptions as _desc

# One capability check covering everything this notebook needs from a
# freshly synced package. Databricks keeps the interpreter alive, so
# `import nn_classifier` does NOT re-read a changed file -- and the
# failure arrives several cells later as a TypeError about an
# unexpected keyword argument, which reads like a coding mistake
# rather than a stale import.
_missing = []
_sig = inspect.signature(_cl.clean_descriptions_with).parameters
for _kw in ("markers", "missing_token"):
    if _kw not in _sig:
        _missing.append(f"clean_descriptions_with({_kw}=)")
if not hasattr(_desc, "_marker_values"):
    _missing.append("descriptions._marker_values (the PLINE_None fix)")
if not hasattr(_cl, "attach_marker_tokens"):
    _missing.append("attach_marker_tokens")
if "marker_columns" not in _cfgmod.CleaningConfig.__dataclass_fields__:
    _missing.append("CleaningConfig.marker_columns")
if _missing:
    raise RuntimeError(
        f"stale nn_classifier -- missing {_missing}\n"
        f"  loaded from: {_cfgmod.__file__}\n"
        "Re-sync the repo, run dbutils.library.restartPython(), and "
        "re-run this notebook from the top."
    )

if data_cfg.vendor_col != "Global Vendno":
    raise RuntimeError(
        f"DataConfig.vendor_col is {data_cfg.vendor_col!r}, expected "
        "'Global Vendno'. You are running a stale nn_classifier.\n"
        f"  loaded from: {_cfgmod.__file__}\n"
        "Re-sync the repo, then run dbutils.library.restartPython() "
        "and re-run this notebook from the top."
    )
print(f"vendor column: {data_cfg.vendor_col!r}")
print(f"nn_classifier: {_cfgmod.__file__}")

PULL = paths.on_databricks()
raw = (data.run_pull(spark) if PULL                      # noqa: F821
       else pd.read_csv(paths.data_path(RAW_FILE), delimiter="|", dtype="str"))

VENDOR_COL = data_cfg.vendor_col
if VENDOR_COL not in raw.columns:
    raise KeyError(
        f"{VENDOR_COL!r} is not in the pull. Columns: {sorted(raw.columns)}. "
        "The SQL aliases it as `Global Vendno`."
    )
# The three code columns, emitted as tokens alongside the vendor.
MARKERS = {
    "ICSP Prodcat": {"prefix": "CAT_"},
    "UNSPSC":       {"prefix": "UNSPSC", "levels": (2, 4, 6, 8)},
    "Prodline":     {"prefix": "PLINE_"},
}
absent = [c for c in MARKERS if c not in raw.columns]
if absent:
    raise KeyError(
        f"{absent} not in the pull. Columns: {sorted(raw.columns)}. "
        "Re-run with the updated HIERARCHY_SQL in data.py."
    )
print(f"{len(raw):,} rows")

---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-5322733600798616>, line 43
     41     _missing.append("CleaningConfig.marker_columns")
     42 if _missing:
---> 43     raise RuntimeError(
     44         f"stale nn_classifier -- missing {_missing}\n"
     45         f"  loaded from: {_cfgmod.__file__}\n"
     46         "Re-sync the repo, run dbutils.library.restartPython(), and "
     47         "re-run this notebook from the top."
     48     )
     50 if data_cfg.vendor_col != "Global Vendno":
     51     raise RuntimeError(
     52         f"DataConfig.vendor_col is {data_cfg.vendor_col!r}, expected "
     53         "'Global Vendno'. You are running a stale nn_classifier.\n"
   (...)     56         "and re-run this notebook from the top."
     57     )

RuntimeError: stale nn_classifier -- missing ['clean_descriptions_with(missing_token=)', 'descriptions._marker_v

## 1 · Is the vendor worth attaching?

Three numbers, and each can sink the arm on its own. Cheaper to find
now than after an hour of training.

**Rows per vendor** — at 1 it is an identifier, not a supplier, which
is the mistake 11b made. Tens to hundreds is a real vendor.

**Held-out coverage** — of rows in a validation split, the share whose
vendor also appears in training. This is the production question: a new
item from a supplier the model already knows. Low coverage means the
marker is absent exactly when it is needed.

**Categories per vendor** — the one that decides whether it carries
*information*. A vendor spanning 5 categories out of 3,492 is worth a
great deal. One spanning 400 is a noise feature however often it
repeats.

In [0]:
vendor = raw[VENDOR_COL].fillna("").astype(str).str.upper().str.strip()
present = vendor != ""
n_vendors = int(vendor[present].nunique())

# Same split settings the model uses, so these numbers transfer.
rng = np.random.default_rng(42)
order = rng.permutation(len(raw))
n_val = int(len(raw) * 0.1)
idx_va, idx_tr = order[:n_val], order[n_val:]
seen = set(vendor.iloc[idx_tr][vendor.iloc[idx_tr] != ""])
va = vendor.iloc[idx_va]
coverage = float((va.isin(seen) & (va != "")).mean())

print(f"vendor present on   : {present.mean():.1%} of rows")
print(f"distinct vendors    : {n_vendors:,}")
print(f"mean rows per vendor: {present.sum() / max(n_vendors, 1):,.1f}")
print(f"held-out coverage   : {coverage:.1%}")
print()

stop = []
if present.mean() < 0.5:
    stop.append("fewer than half the rows carry a vendor")
if present.sum() / max(n_vendors, 1) < 20:
    stop.append("fewer than 20 rows per vendor -- this is closer to an "
                "identifier than a supplier, which is the 11b mistake")
if coverage < 0.8:
    stop.append("under 80% held-out coverage -- the marker is missing on "
                "the rows that need it most")
for s in stop:
    print(f"  >> {s}")
if not stop:
    print("  Usable: vendors repeat, and are present on held-out rows.")

show(vendor[present].value_counts().head(12).to_frame("rows"),
     "no rows carry a vendor at all -- check the column")


# The same three numbers for each new column. A code with one value
# per row is an identifier, not a class; a code absent on most rows
# cannot help the rows it is absent from; and a code spanning hundreds
# of categories narrows nothing however often it repeats.
rows = []
for col in ["Global Vendno"] + list(MARKERS):
    v = raw[col].fillna("").astype(str).str.strip()
    nonblank = v != ""
    seen = set(v.iloc[idx_tr][v.iloc[idx_tr] != ""]) if "idx_tr" in dir() else None
    n_distinct = int(v[nonblank].nunique())
    sub = raw.loc[nonblank, [col, data_cfg.level_columns[-1]]]
    per = sub.groupby(col)[data_cfg.level_columns[-1]].agg(rows="size",
                                                           cats="nunique")
    big = per[per["rows"] >= 50]
    rows.append({
        "column": col,
        "present": round(float(nonblank.mean()), 4),
        "distinct": n_distinct,
        "rows per value": round(int(nonblank.sum()) / max(n_distinct, 1), 1),
        "median cats (>=50 rows)": int(big["cats"].median()) if len(big) else None,
        "rows covered by those": round(float(big["rows"].sum() / len(raw)), 4),
    })
codes = pd.DataFrame(rows)
display(codes)
print("Vendor scored 55 rows/value and a median of 22 categories, and was")
print("worth +0.0247. Anything with a tighter median should be worth more.")

In [0]:
# Does knowing the vendor narrow the category?
raw = raw.copy()
raw["_vendor"] = vendor

sel = raw[raw["_vendor"] != ""]
per_vendor = (sel.groupby("_vendor")[data_cfg.level_columns[-1]]
              .agg(rows="size", categories="nunique")
              .sort_values("rows", ascending=False))

if per_vendor.empty:
    raise RuntimeError(
        "No row carries a vendor. Check the column and the pull before "
        "going further -- nothing below will mean anything."
    )

# The size distribution first, because the useful cut-off depends on
# it. An earlier version hard-coded 50 rows and produced an empty
# frame, which on Databricks fails with CANNOT_INFER_EMPTY_SCHEMA
# rather than saying "no vendor is that large".
q = per_vendor["rows"].quantile([0.5, 0.75, 0.9, 0.99]).round(1)
print("rows per vendor")
print(f"  median {q.loc[0.5]:>8.1f}   p75 {q.loc[0.75]:>8.1f}   "
      f"p90 {q.loc[0.9]:>8.1f}   p99 {q.loc[0.99]:>8.1f}   "
      f"max {per_vendor['rows'].max():,.0f}")
print(f"  {len(per_vendor):,} vendors over {len(sel):,} rows")
print()

# Big enough to learn from, chosen from the data rather than assumed:
# the largest of 50 / p90 / 5 that still leaves at least 20 vendors.
for candidate in (50, int(q.loc[0.9]), 10, 5, 1):
    big = per_vendor[per_vendor["rows"] >= candidate]
    if len(big) >= 20:
        MIN_ROWS = candidate
        break
else:
    MIN_ROWS, big = 1, per_vendor

print(f"vendors with >= {MIN_ROWS} rows: {len(big):,} "
      f"covering {big['rows'].sum() / len(raw):.1%} of all rows")
show(big.head(15))

med = big["categories"].median()
print()
print(f"median categories per such vendor: {med:.0f} of 3,492")
if MIN_ROWS < 10:
    print("  >> Vendors are small. Most will be pruned by min_token_count=5,")
    print("     so the marker will only reach a minority of rows.")
if med > 200:
    print("  >> The typical vendor spans 200+ categories. Weak signal;")
    print("     expect little gain and consider stopping here.")
elif MIN_ROWS >= 10 and med <= 200:
    print("  Vendors are large enough and concentrate. Worth building.")

## 2 · Clean, with the vendor marker

The marker is prepended *after* cleaning, so the regex pass and the
token filter never see it and cannot mangle it.

A note on the wider point about product codes: with
`filter_to_known_tokens=False`, part numbers inside the description
itself do reach the vectorizer. Most are pruned —
`min_token_count=5` drops anything appearing fewer than five times —
so genuinely unique auto-generated codes will disappear on their own.
Worth re-checking `nnz per row` after the change lands; a sharp drop
would mean the model had been leaning on them more than expected.

In [0]:
# ---------------------------------------------------------------------
# Null codes.
#
# True  (default) -- a null vendor, prodcat, unspsc or prodline emits
#                    NO token. The row simply carries fewer markers.
# False           -- emit the token carrying whatever the column held.
#                    Kept only to reproduce a pre-fix run; this is
#                    what produced PLINE_None.
#
# Neither setting produces a CAT_MISSING or UNSPSC2_MISSING token.
# Those appear only if a specific marker is given an explicit
# "missing" key in MARKERS, which nothing here does:
#
#     "Prodline": {"prefix": "PLINE_", "missing": "NONE"}
#
# That is a per-column opt-in for the one case where absence might
# carry signal -- Prodline comes from a LEFT JOIN on a single
# warehouse, so a null is a structural fact about the item. Leave it
# off unless you have measured that missingness correlates with the
# category.
# ---------------------------------------------------------------------
OMIT_MISSING_MARKERS = True

# No global missing token, ever. Per-marker "missing" keys still win.
MISSING_TOKEN = None
print(f"null codes -> {'omitted' if OMIT_MISSING_MARKERS else 'emitted as-is'}")


clean_cfg = CleaningConfig(
    strip_leading_code=True, normalize_misspellings=True, expand_abbrevs=True,
    filter_to_known_tokens=False,
    attach_vendor=True,
)
filter_cfg = FilterConfig(min_class_count=6, min_path_count=2)

lookup = data.build_hierarchy_lookup(raw, data_cfg.level_columns,
                                     data_cfg.name_columns)
assert LOOKUP_FILE != "hierarchy_lookup.csv"
data.save_table(lookup, LOOKUP_FILE, delimiter=",")

cleaner = cleaning.cleaner_from_config(clean_cfg)
cleaned = cleaning.clean_descriptions_with(
    raw, cleaner,
    column=data_cfg.text_col,
    vendor_col="_vendor",
    vendor_prefix=clean_cfg.vendor_token_prefix,
    markers=MARKERS,                     # <- the three code columns
    missing_token=MISSING_TOKEN,
    drop_blank=True,
)
marked = cleaned[data_cfg.text_col].str.startswith(clean_cfg.vendor_token_prefix)
print(f"rows carrying a marker: {marked.mean():.1%}")
show(cleaned[["FullDesc_orig", data_cfg.text_col]].sample(8, random_state=0))


# Every emitted prefix, not one per column. A hierarchical code
# produces a token per level, and an earlier version of this check
# only looked at the first -- so levels 4, 6 and 8 were unverified.
expected = [("vendor", clean_cfg.vendor_token_prefix)]
for _col, _spec in MARKERS.items():
    _pre = str(_spec.get("prefix", ""))
    _lv = _spec.get("levels")
    if _lv:
        expected += [(f"{_col} L{_n}", f"{_pre}{_n}_") for _n in _lv]
    else:
        expected.append((_col, _pre))

missing = []
for _name, _pre in expected:
    _hit = cleaned[data_cfg.text_col].str.contains(rf"(?:^|\s){_pre}", regex=True)
    _tok = (cleaned.loc[_hit, data_cfg.text_col]
            .str.extract(rf"(?:^|\s)({_pre}\S+)", expand=False))
    print(f"  {_name:<18} {_pre:<11} {_hit.mean():6.1%} of rows, "
          f"{_tok.nunique():>7,} distinct tokens")
    if _hit.mean() == 0:
        missing.append(_name)

if missing:
    raise RuntimeError(
        f"{missing} produced no tokens in the cleaned text. The config "
        "would still record them, so 13b's assertion would pass and the "
        "model would train without them. Check that markers=MARKERS is "
        "passed to clean_descriptions_with above, and that the columns "
        "are populated."
    )

low = [n for n, pre in expected
       if 0 < cleaned[data_cfg.text_col].str.contains(
           rf"(?:^|\s){pre}", regex=True).mean() < 0.25]
if low:
    print()
    print(f"  >> {low} reach under a quarter of rows. They can only help "
          f"that share; judge their value against that, not against the "
          f"headline accuracy.")


In [0]:
levels = list(data_cfg.level_columns)
cleaned = data.drop_null_rows(cleaned, data_cfg.text_col, levels)
cleaned = cleaning.drop_rare_classes(cleaned, levels[-1], filter_cfg.min_class_count)
cleaned = cleaning.drop_rare_paths(cleaned, levels, filter_cfg.min_path_count)
print(f"{len(cleaned):,} rows kept  (the shared run kept 600,606)")

## 3 · Save

In [0]:
import joblib

assert CLEAN_FILE != "cleaned_data.txt"
assert str(paths.output_dir()) == ARM_OUTPUT_DIR

data.save_table(cleaned, CLEAN_FILE)
joblib.dump(cleaner, paths.output_path("cleaner.joblib"), compress=3)

cfg = RunConfig()
cfg.data = data_cfg
cfg.data.path = paths.data_path(CLEAN_FILE)
cfg.data.vendor_col = "_vendor"
clean_cfg.marker_columns = MARKERS
cfg.cleaning = clean_cfg
cfg.filters = filter_cfg
cfg.filters.min_token_count = 5
cfg.sparse.kind = "count"
cfg.sparse.binary = True
cfg.sparse.ngram_range = (1, 2)
cfg.sparse.unordered_bigrams = True
cfg.sparse.min_df = 3
cfg.sparse.max_features = None
cfg.notes = ("markers: vendor + " + ", ".join(MARKERS)
             + ". Cleaning rules unchanged.")

cfg.validate()
save_config(cfg)
print()
print(f"marker column: {VENDOR_COL}  ->  cfg.data.vendor_col = '_vendor'")
print("run 12b next")